Configure input paths, model settings, and the output directory.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'SV-Integration').is_dir() and (ROOT.parent / 'SV-Integration').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'SV-Integration').is_dir() and (ROOT / 'SV-velo').is_dir(), 'Open this notebook from the project directory or set ROOT explicitly.'

# Supply the actual reference and spatial input files.
SC_PATH = None     # e.g. ROOT / 'reference_sc_raw.h5ad'
SP_PATH = None     # e.g. ROOT / 'spatial_expression.h5ad'
CELL_TYPE_KEY = 'cell_type'
SPATIAL_KEY = 'spatial'  # Use 'X_spatial' if that is your coordinate key.
SC_EXPRESSION_LAYER = None  # None uses X; otherwise specify a reference expression layer.
SP_EXPRESSION_LAYER = None  # None uses X; otherwise specify a spatial expression layer.
TRAINING_GENES = None       # None uses nonzero shared genes; optionally supply a gene list.
N_PROGRAMS = 8  # Set to ceil(number of Leiden clusters * 1.2) for your dataset.
INTEGRATION_EPOCHS = 1000
VELOCITY_EPOCHS = 200
LEARNING_RATE = 0.01
OUTPUT = ROOT / 'sv_integration_velocity_output'

Import the local packages and load and validate the input datasets.

In [ ]:
import sys, types, importlib, importlib.util
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scvelo as scv
import torch, pyro
import matplotlib.pyplot as plt
from scipy import sparse

def local_package(name, folder, initialize=True):
    folder = Path(folder)
    if initialize:
        spec = importlib.util.spec_from_file_location(name, folder / '__init__.py', submodule_search_locations=[str(folder)])
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
    else:
        module = types.ModuleType(name)
        module.__path__ = [str(folder)]
        sys.modules[name] = module
    return module

# Load only the integration training interface.
local_package('svintegration_local', ROOT / 'SV-Integration', initialize=False)
integration = importlib.import_module('svintegration_local.mapping_utils')
svvelo = local_package('svvelo_local', ROOT / 'SV-velo')
pyro.set_rng_seed(0)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
sc.settings.set_figure_params(dpi=100, frameon=False)

if SC_PATH is None or SP_PATH is None:
    raise ValueError('Set SC_PATH and SP_PATH to the reference counts and spatial expression files.')
reference = ad.read_h5ad(Path(SC_PATH))
spatial = ad.read_h5ad(Path(SP_PATH))
assert reference.obs_names.is_unique and spatial.obs_names.is_unique
assert reference.var_names.is_unique and spatial.var_names.is_unique, 'Resolve duplicate gene names explicitly before proceeding.'
assert CELL_TYPE_KEY in reference.obs, f'Reference data are missing {CELL_TYPE_KEY}'
assert reference.obs[CELL_TYPE_KEY].notna().all(), 'Cell type annotations must not contain missing values.'
assert SPATIAL_KEY in spatial.obsm, f'Spatial data are missing obsm[{SPATIAL_KEY!r}]'
xy = np.asarray(spatial.obsm[SPATIAL_KEY], dtype=float)
assert xy.shape == (spatial.n_obs, 2) and np.isfinite(xy).all()
for layer in ['spliced', 'unspliced']:
    assert layer in reference.layers, f'Reference data are missing raw {layer} counts'
    x = reference.layers[layer]
    values = x.data if sparse.issparse(x) else np.asarray(x)
    assert np.isfinite(values).all() and (values >= 0).all()
    assert np.allclose(values, np.rint(values), atol=1e-6, rtol=0), 'Provide raw integer counts; do not round normalized or smoothed values.'
if SC_EXPRESSION_LAYER is not None: reference.X = reference.layers[SC_EXPRESSION_LAYER].copy()
if SP_EXPRESSION_LAYER is not None: spatial.X = spatial.layers[SP_EXPRESSION_LAYER].copy()
for obj in [reference, spatial]:
    values = obj.X.data if sparse.issparse(obj.X) else np.asarray(obj.X)
    assert np.isfinite(values).all() and (values >= 0).all(), 'Integration expression must be finite and nonnegative.'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Reference cells:', reference.shape, 'Spatial data:', spatial.shape, 'Device:', DEVICE)

Run SV-Integration to obtain the cell-to-spatial-location mapping.

In [ ]:
integration.pp_adatas(reference, spatial, genes=TRAINING_GENES, gene_to_lowercase=False)
assert len(reference.uns['training_genes']) >= 2, 'At least two nonzero shared genes are required.'
assert np.all(np.asarray(reference.X.sum(axis=1)).ravel() > 0)
assert np.all(np.asarray(spatial.X.sum(axis=1)).ravel() > 0)
mapping = integration.map_cells_to_space(
    reference, spatial, cluster_label=CELL_TYPE_KEY, device=DEVICE,
    num_epochs=INTEGRATION_EPOCHS, n_programs=N_PROGRAMS,
    learning_rate=0.1, random_state=0, verbose=True,
)
M = np.asarray(mapping.X, dtype=np.float32)
assert mapping.obs_names.equals(reference.obs_names)
assert mapping.var_names.equals(spatial.obs_names)
assert np.isfinite(M).all() and (M >= 0).all()
assert np.allclose(M.sum(axis=1), 1, atol=1e-4)
mapping.write_h5ad(OUTPUT / 'sv_integration_mapping.h5ad')
print('Integration complete; mapping shape (cells x spatial locations):', M.shape)

Fit SV-velo to raw reference counts and infer RNA velocity.

In [ ]:
genes = list(reference.uns['overlap_genes'])
assert len(genes) >= 2
rna = reference[:, genes].copy()
active = np.asarray((rna.layers['spliced'] + rna.layers['unspliced']).sum(axis=0)).ravel() > 0
rna = rna[:, active].copy()
rna.obsm['X_spatial'] = M @ xy
rna.obsm['svintegration_program_W'] = np.asarray(mapping.obsm['program_W'])
rna.obs[CELL_TYPE_KEY] = pd.Categorical(rna.obs[CELL_TYPE_KEY])
K = min(N_PROGRAMS, rna.n_vars)
pyro.set_rng_seed(0)
svvelo.HVGK.setup_anndata(rna, cell_type_key=CELL_TYPE_KEY)
model = svvelo.HVGK(rna, n_modules=K, initialization='expression',
                    initialization_seed=0, spatial_config={'enabled': False})
model.train(max_epochs=VELOCITY_EPOCHS, accelerator='gpu' if torch.cuda.is_available() else 'cpu',
            device=0 if torch.cuda.is_available() else 'auto', batch_size=rna.n_obs,
            train_size=1.0, early_stopping=False, lr=LEARNING_RATE)

# Use joint posterior draws for velocity and its paired biological expression.
posterior = model.sample_posterior(num_samples=30, batch_size=rna.n_obs,
    return_sites=['velocity', 'expression_biological_s', 'expression_biological_u'], return_samples=False)
means = posterior['post_sample_means']
for site in ['velocity', 'expression_biological_s', 'expression_biological_u']:
    value = np.asarray(means[site], dtype=np.float32)
    assert value.shape == rna.shape and np.isfinite(value).all()
    rna.layers[site] = value
model.save(str(OUTPUT / 'sv_velo_model'), overwrite=True)
print('RNA velocity inferred:', rna.layers['velocity'].shape)

Project RNA velocity into spatial locations, visualize it, and save the results.

In [ ]:
mass = M.sum(axis=0)
keep = mass > 0
weights = M[:, keep].T / mass[keep, None]
view = ad.AnnData(X=(weights @ rna.layers['expression_biological_s']).astype('float32'),
                  obs=spatial.obs.loc[spatial.obs_names[keep]].copy(), var=rna.var.copy())
view.obsm['X_spatial'] = xy[keep]
view.layers['Ms'] = view.X.copy()
view.layers['Mu'] = (weights @ rna.layers['expression_biological_u']).astype('float32')
view.layers['velocity'] = (weights @ rna.layers['velocity']).astype('float32')
view.obs['mapping_mass'] = mass[keep]
composition = weights @ pd.get_dummies(rna.obs[CELL_TYPE_KEY], dtype=float).to_numpy()
labels = pd.get_dummies(rna.obs[CELL_TYPE_KEY]).columns.astype(str)
view.obs['mapped_cell_type'] = pd.Categorical(labels.to_numpy()[composition.argmax(axis=1)])
view.obsm['mapped_cell_type_composition'] = composition
view.uns['velocity_provenance'] = 'SV-velo biological RNA derivative on reference raw counts; SV-Integration mass-normalized spatial projection; not physical migration'

# Build the expression neighbor graph for two-dimensional velocity visualization.
assert min(view.shape) >= 3, 'Visualization requires at least three spatial locations and three genes.'
features = view.copy()
features.X = np.log1p(np.asarray(view.layers['Ms']))
sc.pp.pca(features, n_comps=min(30, features.n_vars-1, features.n_obs-1))
sc.pp.neighbors(features, n_neighbors=min(30, features.n_obs-1))
view.obsm['X_pca'] = features.obsm['X_pca']
view.obsp['distances'] = features.obsp['distances']
view.obsp['connectivities'] = features.obsp['connectivities']
view.uns['neighbors'] = features.uns['neighbors']
scv.tl.velocity_graph(view, vkey='velocity', xkey='Ms', n_jobs=1)
scv.tl.velocity_embedding(view, basis='spatial', vkey='velocity')
ax = scv.pl.velocity_embedding_stream(view, basis='spatial', vkey='velocity',
    color='mapped_cell_type', legend_loc='right margin', size=80,
    title='SV-Integration → SV-velo', show=False)
ax.grid(True, color='#d8d8d8', linewidth=.65)
ax.set_rasterized(True)
for ext in ['png', 'pdf']:
    ax.figure.savefig(OUTPUT / f'spatial_rna_velocity.{ext}', dpi=300, bbox_inches='tight')
plt.show()
rna.write_h5ad(OUTPUT / 'reference_rna_velocity.h5ad')
view.write_h5ad(OUTPUT / 'spatial_rna_velocity.h5ad')
print('Output directory:', OUTPUT)